# Mission 9: Pairs Trading & Statistical Arbitrage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_09_pairs_trading/notebook.ipynb)

**Advanced elective.** Every strategy so far has been *cross-sectional* — rank many stocks at one
moment. Pairs trading is the canonical *time-series* alternative: find two assets whose prices are
tied together by a long-run equilibrium, and bet that temporary divergences snap back. It's the
textbook example of **statistical arbitrage** — and a textbook example of how a relationship that
looks ironclad in-sample can quietly fall apart.

**Learning objectives**
- Distinguish **correlation** from **cointegration** — and why only the latter gives a tradeable spread
- Test for cointegration (OLS hedge ratio + ADF / Engle–Granger) and form a stationary spread
- Trade the spread with a **z-score** entry/exit rule and evaluate it
- See **spurious cointegration**: how scanning many pairs manufactures false equilibria that break out of sample

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q "git+https://github.com/convexpi/lab.git@b3006123f7bf455d0dbb1cb03a87dd8bd3e5bed0" statsmodels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
rng = np.random.default_rng(7)
print('Ready.')

---
## Part 1: Correlation is not cointegration

Two stock prices can be highly *correlated* (they go up and down together day to day) yet drift apart
forever. What pairs trading needs is stronger: **cointegration** — the two prices share a common
stochastic trend, so a particular linear combination of them (the **spread**) is *stationary*: it
wanders around a fixed mean and keeps returning to it.

Let's manufacture one of each. A cointegrated pair (A, B) shares a common random-walk factor; two
independent random walks are our control.

In [ ]:
T = 800
common = np.cumsum(rng.normal(0, 1, T))            # a shared stochastic trend
A = 50 + common + rng.normal(0, 1, T)              # both prices ride `common`...
B = 20 + 0.8 * common + rng.normal(0, 1, T)        # ...B with sensitivity (beta) 0.8

X = np.cumsum(rng.normal(0, 1, T))                 # control: two INDEPENDENT random walks
Y = np.cumsum(rng.normal(0, 1, T))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(A, label='A'); ax[0].plot(B, label='B'); ax[0].set_title('Cointegrated pair'); ax[0].legend()
ax[1].plot(X, label='X'); ax[1].plot(Y, label='Y'); ax[1].set_title('Independent random walks'); ax[1].legend()
plt.tight_layout(); plt.show()

Both panels *look* like prices that move together. The eye can't tell them apart — which is exactly
why we need a statistical test, not a chart.

---
## Part 2: Testing for cointegration

The recipe (Engle–Granger): regress one price on the other to get a **hedge ratio** β, form the
spread `B − βA`, and test whether that spread is **stationary** with an Augmented Dickey–Fuller (ADF)
test. A low p-value means "the spread reverts" — the pair is cointegrated.

In [ ]:
def hedge_and_spread(a, b):
    beta = np.polyfit(a, b, 1)[0]      # OLS slope: how many units of A hedge one unit of B
    return beta, b - beta * a

for name, (p, q) in {'cointegrated (A,B)': (A, B), 'independent (X,Y)': (X, Y)}.items():
    beta, spread = hedge_and_spread(p, q)
    adf_p = adfuller(spread)[1]                 # ADF on the spread
    eg_p = coint(p, q)[1]                        # Engle-Granger one-shot test
    verdict = 'COINTEGRATED' if eg_p < 0.05 else 'not cointegrated'
    print(f"{name:22} beta={beta:5.2f}  ADF p(spread)={adf_p:7.4f}  Engle-Granger p={eg_p:7.4f}  -> {verdict}")

In [ ]:
# The spread of a true pair is stationary (mean-reverting); the 'spread' of independents wanders.
_, sp_AB = hedge_and_spread(A, B)
_, sp_XY = hedge_and_spread(X, Y)
plt.figure(figsize=(9, 3))
plt.plot(sp_AB - sp_AB.mean(), label='A,B spread (stationary)')
plt.plot(sp_XY - sp_XY.mean(), label='X,Y spread (wanders)', alpha=0.8)
plt.axhline(0, color='k', lw=0.8); plt.legend(); plt.title('A tradeable spread reverts to its mean')
plt.tight_layout(); plt.show()

**Exercise 2.1** — Lower B's sensitivity to the common trend toward zero (e.g. `0.8 → 0.1`) and add
more idiosyncratic noise. At what point does the Engle–Granger test stop calling the pair
cointegrated? Cointegration is a matter of degree, and weak cointegration is barely tradeable after costs.

---
## Part 3: Trading the spread

Once you trust the spread, the rule is simple. Standardise it into a rolling **z-score**; when the
spread stretches far from its mean (`|z| > entry`), bet on reversion — short the spread when it's
high, long it when it's low — and close out as it returns (`|z| < exit`).

In [ ]:
def backtest_pair(a, b, beta, entry=2.0, exit=0.5, lookback=60):
    s = b - beta * a
    z = np.full(len(s), np.nan)
    for t in range(lookback, len(s)):
        win = s[t - lookback:t]
        z[t] = (s[t] - win.mean()) / (win.std() + 1e-9)
    ds = np.diff(s)                                   # spread P&L per unit of spread position
    pos = 0; pnl = []; states = []
    for t in range(lookback, len(s) - 1):
        if pos == 0:
            if z[t] > entry: pos = -1                 # spread too high -> short it
            elif z[t] < -entry: pos = +1              # spread too low  -> long it
        elif abs(z[t]) < exit:
            pos = 0                                    # reverted -> flat
        pnl.append(pos * ds[t]); states.append(pos)
    pnl = np.array(pnl)
    sharpe = pnl.mean() / (pnl.std() + 1e-9) * np.sqrt(252)
    return z, np.array(states), pnl, sharpe

beta, _ = hedge_and_spread(A, B)
z, states, pnl, sharpe = backtest_pair(A, B, beta)
print(f"hedge ratio beta : {beta:.2f}")
print(f"pair Sharpe      : {sharpe:.2f}")
print(f"total spread P&L : {pnl.sum():.1f}   (round-trips: {int((np.diff(states) != 0).sum())})")

fig, ax = plt.subplots(2, 1, figsize=(9, 4.6), sharex=True)
ax[0].plot(z); ax[0].axhline(2, color='C3', ls='--', lw=1); ax[0].axhline(-2, color='C3', ls='--', lw=1)
ax[0].axhline(0, color='k', lw=0.6); ax[0].set_ylabel('spread z-score'); ax[0].set_title('Entry at |z|>2, exit near 0')
ax[1].plot(np.cumsum(pnl)); ax[1].set_ylabel('cumulative P&L'); ax[1].set_xlabel('day')
plt.tight_layout(); plt.show()

**Exercise 3.1** — Sweep `entry` over `[1.0, 1.5, 2.0, 2.5, 3.0]`. Wider entry thresholds trade less
often but each trade is a stronger signal. Where is the Sharpe best, and what happens to the number
of round-trips? (Then remember Mission 8: more round-trips means more transaction costs.)

---
## Part 4: The danger — spurious cointegration

Here's the trap that ends most pairs strategies. If you **scan thousands of pairs** looking for
cointegration, you'll find plenty — *by chance*. At a 5% significance level, roughly 5% of unrelated
pairs will "pass" in-sample. Those aren't equilibria; they're noise that briefly rhymed. Out of
sample, they fall apart.

We scan a realistic synthetic universe, flag every pair that looks cointegrated **in-sample**, then
check how many still pass on the **holdout**.

In [ ]:
from convexpi.lab import SyntheticMarket
mkt = SyntheticMarket(n_assets=60, n_days=1000, seed=42)
P_is = mkt.prices('train')      # in-sample window
P_oos = mkt.prices('test')      # holdout window (the honest test)
N = P_is.shape[1]

is_hits, survivors, broken_example = [], [], None
for i in range(N):
    for j in range(i + 1, N):
        if coint(P_is[:, i], P_is[:, j])[1] < 0.05:        # 'cointegrated' in-sample
            is_hits.append((i, j))
            if coint(P_oos[:, i], P_oos[:, j])[1] < 0.05:  # ...still cointegrated OOS?
                survivors.append((i, j))
            elif broken_example is None:
                broken_example = (i, j)

checked = N * (N - 1) // 2
print(f"pairs scanned          : {checked}")
print(f"'cointegrated' in-sample: {len(is_hits)}  ({len(is_hits)/checked:.1%} — near the 5% you'd expect by chance)")
print(f"still cointegrated OOS  : {len(survivors)}  ({len(survivors)/max(len(is_hits),1):.0%} of the in-sample hits)")
print("\nMost 'discovered' pairs are spurious. Finding cointegration is easy; finding it OOS is the job.")

In [ ]:
# A spurious pair: its spread looks mean-reverting in-sample, then drifts on the holdout.
i, j = broken_example
beta_is = np.polyfit(P_is[:, i], P_is[:, j], 1)[0]
sp_is = P_is[:, j] - beta_is * P_is[:, i]
sp_oos = P_oos[:, j] - beta_is * P_oos[:, i]     # apply the IN-SAMPLE hedge ratio out of sample
plt.figure(figsize=(9, 3))
plt.plot(np.arange(len(sp_is)), sp_is - sp_is.mean(), label='in-sample spread (looks tradeable)')
plt.plot(np.arange(len(sp_is), len(sp_is) + len(sp_oos)), sp_oos - sp_is.mean(),
         color='C3', label='holdout spread (drifts away)')
plt.axvline(len(sp_is), color='k', ls=':', lw=1); plt.axhline(0, color='k', lw=0.6)
plt.legend(); plt.title(f'Spurious pair (assets {i},{j}): the equilibrium was never real')
plt.tight_layout(); plt.show()

This is Mission 1's lesson wearing a time-series costume. The defences are the same: **out-of-sample
validation** (does the pair still cointegrate on the holdout?), **multiple-testing awareness** (you
tested thousands of pairs — adjust your threshold, e.g. Bonferroni/FDR), and an **economic prior**
(pairs with a real reason to be linked — same sector, same supply chain — are far likelier to persist
than pairs your scan stumbled on).

---
## Challenge

Build an honest pairs-trading pipeline on the synthetic universe:

1. Find candidate pairs that cointegrate **in-sample**, but keep only those that **also** cointegrate
   on a validation slice (a true out-of-sample filter).
2. Backtest the z-score rule on the survivors and report Sharpe **net of costs** (reuse Mission 8 — a
   pairs strategy can be high-turnover).
3. Compare the net Sharpe of the validated survivors against the full in-sample set. How much of the
   "edge" was spurious?

Write it up and publish to **[/projects](https://convexpi.ai/projects)** — a clear demonstration of
spurious-cointegration discipline is exactly the kind of result the leaderboard rewards.